# Pogoda i klimat w Polsce 1951–2025

Notebook krok po kroku: **pobranie → czyszczenie → analiza → wykresy → wnioski**.

Odpowiadamy na cztery pytania:
1. Czy jest cieplej niż kiedyś?
2. Ile jest dni upałów i mrozów?
3. Jak zmieniły się opady i śnieg?
4. Czy zimy są krótsze?

Miasta: Warszawa, Kraków, Gdańsk, Wrocław, Poznań. Dane: [Open-Meteo Historical API](https://open-meteo.com/en/docs/historical-weather-api)
(reanaliza ERA5, czyli model pogody dopasowany do historycznych pomiarów; wartości są nieco wygładzone względem pojedynczej stacji).

> Jak uruchomić: `uv run jupyter lab` w folderze projektu, potem *Run → Run All Cells*.

## 0. Przygotowanie

Wspólne funkcje leżą w `src/pogoda.py` (dane i wskaźniki) oraz `src/charts.py` (wykresy).
Notebook i raport HTML używają tego samego kodu, więc liczby zawsze się zgadzają.
Dodajemy folder `src` do ścieżki, żeby Python znalazł te moduły.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
import plotly.express as px
import plotly.io as pio

import charts
import pogoda

# Wykresy w notebooku: interaktywne, plotly.js ładowany z internetu (mniejszy plik).
pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.precision", 2)

## 1. Pobranie danych

Funkcja `fetch_all()` pyta API o dane **dzienne** (temperatura max/min/średnia, suma opadu, opad śniegu)
dla każdego miasta i zapisuje odpowiedź do `data/raw/<miasto>.csv`.

Jeśli plik już istnieje, nie pobiera go ponownie. To tzw. **cache**: lokalna kopia, dzięki której
nie obciążamy API i notebook działa bez internetu. Żeby wymusić ponowne pobranie: `fetch_all(force=True)`.

In [2]:
pogoda.fetch_all();

Warszawa   (z cache) -> warszawa.csv
Kraków     (z cache) -> krakow.csv
Gdańsk     (z cache) -> gdansk.csv
Wrocław    (z cache) -> wroclaw.csv
Poznań     (z cache) -> poznan.csv


Tak wygląda pojedyncze zapytanie, gdybyś chciał(a) je wpisać w przeglądarce
(to zwykły adres URL z parametrami po `?`):

In [3]:
lat, lon = pogoda.CITIES["Warszawa"]
print(f"{pogoda.API_URL}?latitude={lat}&longitude={lon}"
      f"&start_date=2025-12-01&end_date=2025-12-03&daily=temperature_2m_max,precipitation_sum")

https://archive-api.open-meteo.com/v1/archive?latitude=52.23&longitude=21.01&start_date=2025-12-01&end_date=2025-12-03&daily=temperature_2m_max,precipitation_sum


## 2. Wczytanie i czyszczenie

`load_raw()` skleja 5 plików CSV w jedną długą tabelę z kolumną `city` (format *long*: jeden wiersz = jedno miasto w jednym dniu).
`clean()`:
- usuwa ewentualne duplikaty i sortuje,
- dodaje kolumny pomocnicze `year`, `month` oraz `season` (sezon zimowy „2024” = lipiec 2023 – czerwiec 2024,
  żeby zima z przełomu roku nie była rozcięta na dwa lata).

In [4]:
raw = pogoda.load_raw()
df = pogoda.clean(raw)
df.head()

,city,date,tmax,tmin,tmean,precip,snow,year,month,season
0,Gdańsk,1951-01-01,-3.5,-13.1,-8.3,0.0,0.00,1951,1,1951
1,Gdańsk,1951-01-02,-1.7,-5.1,-3.3,0.6,0.42,1951,1,1951
2,Gdańsk,1951-01-03,3.1,-1.7,1.2,2.3,0.00,1951,1,1951
3,Gdańsk,1951-01-04,2.4,0.2,0.8,0.2,0.00,1951,1,1951
4,Gdańsk,1951-01-05,0.7,-1.7,-0.7,0.1,0.00,1951,1,1951


Sprawdzamy jakość danych: typy kolumn, braki, zakres dat i liczbę dni na miasto.

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 136970 entries, 0 to 136969
Data columns (total 10 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   city    136970 non-null  category      
 1   date    136970 non-null  datetime64[us]
 2   tmax    136970 non-null  float64       
 3   tmin    136970 non-null  float64       
 4   tmean   136970 non-null  float64       
 5   precip  136970 non-null  float64       
 6   snow    136970 non-null  float64       
 7   year    136970 non-null  int32         
 8   month   136970 non-null  int32         
 9   season  136970 non-null  int32         
dtypes: category(1), datetime64[us](1), float64(5), int32(3)
memory usage: 8.0 MB


In [6]:
print("Braki danych w kolumnach:")
print(df.isna().sum())
df.groupby("city", observed=True)["date"].agg(["min", "max", "count"])

Braki danych w kolumnach:
city      0
date      0
tmax      0
tmin      0
tmean     0
precip    0
snow      0
year      0
month     0
season    0
dtype: int64


,min,max,count
city,,,
Warszawa,1951-01-01,2025-12-31,27394
Kraków,1951-01-01,2025-12-31,27394
Gdańsk,1951-01-01,2025-12-31,27394
Wrocław,1951-01-01,2025-12-31,27394
Poznań,1951-01-01,2025-12-31,27394


Szybki test sensowności (*sanity check*): czy wartości mieszczą się w realnym zakresie?
Najgorętszy i najzimniejszy dzień w danych:

In [7]:
df.loc[[df["tmax"].idxmax(), df["tmin"].idxmin()], ["city", "date", "tmax", "tmin"]]

,city,date,tmax,tmin
80922,Poznań,2022-07-21,37.3,21.5
122737,Wrocław,1987-01-13,-25.1,-33.7


Dane z reanalizy są kompletne (0 braków), więc czyszczenie jest proste.
Gdybyśmy użyli danych IMGW ze stacji, tu byłaby największa praca: luki, zmiany lokalizacji stacji, kodowanie znaków.

## 3. Analiza: wskaźniki roczne

`yearly()` zamienia ~137 tys. dni w ~375 wierszy (5 miast × 75 lat). Dla każdego roku liczy:

| kolumna | znaczenie |
|---|---|
| `tmean` | średnia roczna temperatura |
| `anomaly` | odchylenie od średniej z lat 1961–1990 (tzw. **norma klimatyczna**) |
| `precip` | roczna suma opadów (mm) |
| `hot_days` | dni upalne: Tmax ≥ 30 °C |
| `ice_days` | dni mroźne: Tmax < 0 °C (cały dzień poniżej zera) |
| `cold_nights` | bardzo mroźne noce: Tmin ≤ −10 °C |
| `snow_days` | dni z opadem śniegu ≥ 1 cm |

Kluczowy mechanizm to `groupby(["city", "year"])`: podziel tabelę na grupy i policz coś w każdej z nich.

In [8]:
yearly = pogoda.yearly(df)
yearly.tail()

,city,year,tmean,tmax_mean,tmin_mean,precip,hot_days,ice_days,cold_nights,snow_days,anomaly
370,Poznań,2021,9.30,13.15,5.47,671.4,7,23,12,21,0.80
371,Poznań,2022,10.51,14.72,6.31,529.3,19,9,0,8,2.01
372,Poznań,2023,10.65,14.61,6.60,756.2,5,12,0,18,2.15
373,Poznań,2024,11.55,15.48,7.61,581.2,18,12,2,7,3.05
374,Poznań,2025,10.15,14.18,6.22,563.5,8,11,4,9,1.65


Najpierw prosty wykres w **plotly express** (`px`), czyli najkrótszym sposobem na wykres z tabeli.
Jedna linijka, a wykres już jest interaktywny (najedź kursorem, przybliż, kliknij legendę):

In [9]:
px.line(yearly, x="year", y="tmean", color="city", title="Średnia roczna temperatura (°C)")

Linie się nakładają, a rok do roku dużo „szumu”. Dlatego dalej:
- uśredniamy 5 miast (to dobre przybliżenie „Polski”, bo miasta zmieniają się bardzo podobnie),
- dodajemy **średnią kroczącą 10-letnią** (każdy punkt = średnia z 10 lat wokół niego), która pokazuje trend zamiast szumu.

Gotowe, dopracowane wykresy są w `src/charts.py`.

### Pytanie 1: Czy jest cieplej?

In [10]:
charts.anomaly_bars(yearly)

**Trend liniowy** to nachylenie prostej dopasowanej metodą najmniejszych kwadratów (`numpy.polyfit`).
Podajemy go w °C na dekadę:

In [11]:
trend = pogoda.trends(yearly, ["tmean", "hot_days", "ice_days", "precip", "snow_days"])
trend.columns = ["°C / dekadę", "dni upału / dek.", "dni mroźne / dek.", "opad mm / dek.", "dni ze śniegiem / dek."]
trend

,°C / dekadę,dni upału / dek.,dni mroźne / dek.,opad mm / dek.,dni ze śniegiem / dek.
Warszawa,0.33,1.16,-4.03,2.23,-1.16
Kraków,0.38,1.22,-4.15,6.46,-1.58
Gdańsk,0.33,0.39,-3.90,5.55,-0.82
Wrocław,0.34,1.11,-3.42,0.74,-0.66
Poznań,0.32,1.10,-3.28,2.30,-0.97


In [12]:
pl = charts.poland_mean(yearly).set_index("year")
periods = {"1961–1990": (1961, 1990), "1996–2025": (1996, 2025), "2016–2025": (2016, 2025)}
compare = pd.DataFrame({name: pl.loc[a:b].mean() for name, (a, b) in periods.items()})
compare.loc[["tmean", "anomaly", "hot_days", "ice_days", "cold_nights", "snow_days", "precip"]].round(2)

,1961–1990,1996–2025,2016–2025
tmean,8.12,9.47,10.26
anomaly,0.00,1.35,2.14
hot_days,1.37,5.56,8.42
ice_days,39.14,25.21,14.02
cold_nights,15.89,9.19,4.04
snow_days,20.55,16.05,12.72
precip,682.39,674.70,720.52


In [13]:
charts.city_dumbbell(yearly)

In [14]:
warm = pl["anomaly"].sort_values(ascending=False)
print("Najcieplejsze lata:", ", ".join(f"{y} ({v:+.1f} °C)" for y, v in warm.head(5).items()))
print("Lata od 1991 chłodniejsze od normy:", list(pl.loc[1991:].query("anomaly < 0").index))

Najcieplejsze lata: 2024 (+3.3 °C), 2019 (+2.8 °C), 2023 (+2.5 °C), 2018 (+2.4 °C), 2020 (+2.4 °C)
Lata od 1991 chłodniejsze od normy: [1996, 2010]


### Pytanie 2: Upały i mrozy

In [15]:
charts.yearly_counts(yearly, "hot_days", "Dni upalne w roku (Tmax ≥ 30 °C)", charts.ORANGE)

In [16]:
charts.yearly_counts(yearly, "ice_days", "Dni mroźne w roku (Tmax < 0 °C)", charts.BLUE)

In [17]:
yearly.pivot_table(index="city", columns=pd.cut(yearly["year"], [1950, 1990, 2015, 2025],
                   labels=["1951–1990", "1991–2015", "2016–2025"]),
                   values="hot_days", observed=True).round(1)

year,1951–1990,1991–2015,2016–2025
city,,,
Warszawa,1.6,4.6,10.6
Kraków,1.0,4.9,9.4
Gdańsk,0.6,1.8,3.1
Wrocław,2.0,5.7,9.0
Poznań,2.0,4.8,10.0


### Pytanie 3: Opady i śnieg

Uwaga: opady w reanalizie są mniej pewne niż temperatura (deszcz jest bardzo lokalny), więc traktujemy je ostrożniej.

In [18]:
charts.yearly_counts(yearly, "precip", "Roczna suma opadów", charts.BLUE, unit="mm")

In [19]:
charts.monthly_precip(df)

In [20]:
charts.yearly_counts(yearly, "snow_days", "Dni z opadem śniegu ≥ 1 cm", charts.BLUE)

### Pytanie 4: Czy zimy są krótsze?

**Zima termiczna** to okres ze średnią dobową temperaturą poniżej 0 °C, a **lato termiczne** ze średnią powyżej 15 °C.
Pojedyncze dni skaczą, więc najpierw wygładzamy temperaturę średnią kroczącą 7-dniową,
a potem liczymy, ile dni w sezonie jest poniżej 0 °C (zima) i ile w roku powyżej 15 °C (lato).

In [21]:
seasons = pogoda.seasons(df)
seasons.head()

,city,year,winter_days,summer_days
0,Warszawa,1952,71,96
1,Warszawa,1953,93,109
2,Warszawa,1954,82,120
3,Warszawa,1955,82,95
4,Warszawa,1956,82,99


In [22]:
charts.season_lengths(seasons)

In [23]:
ps = charts.poland_mean(seasons).set_index("year")
pd.DataFrame({
    "1962–1990": ps.loc[1962:1990].mean(),
    "2016–2025": ps.loc[2016:2025].mean(),
    "trend / dekadę": pogoda.trends(seasons, ["winter_days", "summer_days"]).mean(),
}).round(1)

,1962–1990,2016–2025,trend / dekadę
winter_days,65.8,29.1,-5.8
summer_days,95.6,123.8,3.9


## 4. Wnioski

1. **Jest wyraźnie cieplej.** Średnia temperatura rośnie o ok. 0,3 °C na dekadę (od 1981 r. jeszcze szybciej, ok. 0,5 °C).
   Ostatnie 10 lat jest o ok. 2 °C cieplejsze niż norma 1961–1990, a od 1991 r. tylko dwa lata były chłodniejsze od normy.
   Wzrost widać we wszystkich pięciu miastach, prawie po równo.
2. **Upałów przybyło, mrozów ubyło.** Dni z Tmax ≥ 30 °C jest kilka razy więcej niż w latach 1961–1990,
   a dni, w których cały czas jest poniżej zera, jest mniej więcej o połowę mniej. Najmniej upałów ma Gdańsk (chłodzi go morze).
3. **Opady: suma bez wyraźnego trendu, za to zmienia się rozkład.** Roczna suma waha się mocno z roku na rok i nie ma wyraźnego kierunku.
   Rozkład w roku się przesuwa, a śniegu jest wyraźnie mniej.
4. **Zimy są dużo krótsze, lata dłuższe.** Zima termiczna skurczyła się mniej więcej o połowę, a lato termiczne wydłużyło się o kilka tygodni.

Dokładne liczby są w tabelach powyżej i w raporcie `report/index.html` (generuje go `uv run python scripts/build_report.py`).

**Ograniczenia:** dane z reanalizy (model, a nie pomiar ze stacji), 5 punktów zamiast całego kraju, trend liniowy to uproszczenie.